In [ ]:
# !pip install yfinance
# !pip install TA-Lib 
# !pip install numpy
# !pip install pandas
# !pip install vectorbt
# !pip install scipy

In [ ]:
import yfinance as yf
import talib
import numpy as np
import pandas as pd
import vectorbt as vbt
import warnings
from scipy import stats
import matplotlib.pyplot as plt
from tqdm import tqdm


In [ ]:
# DOWNLOAD STOCK DATA FROM 2018 USING YFINANCE

# Configuration - Change these variables as needed
TICKER = 'BTC-USD'  # Any ticker symbol (e.g., 'AAPL', 'MSFT', 'GOOGL')
START_DATE = '2017-01-01'
END_DATE = '2024-06-21'

# Download data from start date onwards
DATA_PATH = r"data/QQQ.csv"
stock_data = pd.read_csv(DATA_PATH, index_col=0, parse_dates=True)

stock_data = stock_data.loc[START_DATE:END_DATE]

if not stock_data.empty:
    print(f"Successfully downloaded {len(stock_data)} records for {TICKER} from {START_DATE}")
    print(f"Data range: {stock_data.index.min().date()} to {stock_data.index.max().date()}")
    print("\nFirst 5 rows:")
    print(stock_data.head())
else:
    print(f"Failed to download {TICKER} data from yfinance")

# Display the downloaded data
stock_data


In [ ]:
# PREPARE PRICE SERIES

warnings.filterwarnings("ignore", message="Degrees of freedom <= 0 for slice", category=RuntimeWarning)
warnings.filterwarnings("ignore", message="invalid value encountered in scalar divide", category=RuntimeWarning)

# Expect stock_data and TICKER already exist
def select_close_series(df, ticker):
    if isinstance(df.columns, pd.MultiIndex):
        if ('Close', ticker) in df.columns:
            s = df[('Close', ticker)]
        else:
            cols = [c for c in df.columns if 'Close' in str(c)]
            if not cols:
                raise KeyError("Close not found")
            s = df[cols[0]]
    else:
        try:
            s = df['Close']
        except KeyError:
            s = df['close']
    return s.astype(float).squeeze()

close = select_close_series(stock_data, TICKER)
close.name = 'price'

# Simple split
TRAIN_RATIO = 0.60 
split_idx = int(len(close) * TRAIN_RATIO)
train_close = close.iloc[:split_idx].copy()
val_close   = close.iloc[split_idx:].copy()

print(f"Data ready: train={train_close.index[0].date()} → {train_close.index[-1].date()} | val={val_close.index[0].date()} → {val_close.index[-1].date()}")

MACD CROSSOVER GRID SEARCH - TRAINING SET
----------------------------------------------

This section performs a comprehensive grid search optimization for the **MACD Crossover Strategy** using only the **training data**.

The goal is to find the optimal Fast/Slow/Signal EMA combination that maximizes the Sharpe ratio on unseen data.

**Strategy Logic**: 
- MACD Line = Fast EMA - Slow EMA
- Signal Line = EMA of MACD Line
- Buy when MACD Line crosses above Signal Line
- Sell when MACD Line crosses below Signal Line

---

In [ ]:
# Define Parameter Ranges for MACD Crossover

# MACD periods for crossover strategy
fast_periods = list(range(8, 20, 2))     # Fast EMA (typically 12)
slow_periods = list(range(20, 35, 2))    # Slow EMA (typically 26)
signal_periods = list(range(5, 15, 2))   # Signal EMA (typically 9)



print("Fast EMA Periods (short-term):")
for i, period in enumerate(fast_periods, 1):
    print(f"  {i}. {period} periods")

print("Slow EMA Periods (long-term):")
for i, period in enumerate(slow_periods, 1):
    print(f"  {i}. {period} periods")

print("Signal EMA Periods:")
for i, period in enumerate(signal_periods, 1):
    print(f"  {i}. {period} periods")

# Generate all valid combinations (fast < slow)
macd_combinations = []
for fast in fast_periods:
    for slow in slow_periods:
        for signal in signal_periods:
            if fast < slow:
                macd_combinations.append((fast, slow, signal))

print(f"Generated {len(macd_combinations)} valid MACD combinations")
print("\n📋 First 10 combinations preview:")
for i, (fast, slow, signal) in enumerate(macd_combinations[:10], 1):
    print(f"  {i:2d}. Fast: {fast:2d} | Slow: {slow:2d} | Signal: {signal:2d}")
if len(macd_combinations) > 10:
    print(f"   ... and {len(macd_combinations) - 10} more combinations")

print("Ready to test all combinations on training data!")

In [ ]:
# Initialize MACD Results Collection System

# Create empty list to store all backtest results
grid_search_results = []

print("MACD Results Collection System Initialized")
print(f"   - Will test {len(macd_combinations)} MACD combinations")
print("   - Results will be stored in 'grid_search_results' list")

# Define what metrics we will collect (All TradingView-style metrics)
metrics_to_collect = [
    # Strategy Parameters
    "fast_period",
    "slow_period", 
    "signal_period",
    
    # Return Metrics
    "total_return",
    "annualized_return",
    "total_profit",
    
    # Risk-Adjusted Return Metrics
    "sharpe_ratio",
    "sortino_ratio",
    "calmar_ratio",
    "omega_ratio",
    "information_ratio",
    "tail_ratio",
    "deflated_sharpe_ratio",
    
    # Risk Metrics
    "max_drawdown",
    "volatility",
    "ulcer_index",
    
    # Trade Performance Metrics
    "win_rate",
    "total_trades",
    "avg_trade_duration",
    "expectancy",
    "profit_factor", 
    "sqn",
    
    # Win/Loss Analysis
    "payoff_ratio",
    "largest_win",
    "largest_loss",
    "avg_win_amount",
    "avg_loss_amount",
    "winning_streak",
    "losing_streak",
    
    # Additional Ratios
    "recovery_factor",
    "gain_to_pain_ratio",
    "serenity_index"
]

print("Metrics to collect for each MACD combination:")
for i, metric in enumerate(metrics_to_collect, 1):
    print(f"  {i}. {metric.replace('_', ' ').title()}")

print("Ready to start the MACD grid search!")


In [ ]:
# MACD GRID SEARCH - ALL COMBINATIONS

FREQ = "1D"
price_np = train_close.to_numpy(dtype=float)

def compute_macd(close_series, fast, slow, signal):
    """Compute MACD line and signal line using TA-Lib"""
    macd_line, signal_line, hist = talib.MACD(
        close_series.values, 
        fastperiod=fast, 
        slowperiod=slow, 
        signalperiod=signal
    )
    return pd.Series(macd_line, index=close_series.index), pd.Series(signal_line, index=close_series.index)

def run_macd_backtest(fast, slow, signal, close_series, freq=FREQ, init_cash=100_000):
    """
    Run backtest for MACD crossover strategy.
    Entry: MACD line crosses above signal line
    Exit: MACD line crosses below signal line
    """
    # Compute MACD
    macd_line, signal_line = compute_macd(close_series, fast, slow, signal)
    
    # Generate signals
    macd_series = pd.Series(macd_line, index=close_series.index)
    signal_series = pd.Series(signal_line, index=close_series.index)
    
    # MACD crossover signals
    entries_raw = (macd_series > signal_series) & (macd_series.shift(1) <= signal_series.shift(1))
    exits_raw = (macd_series < signal_series) & (macd_series.shift(1) >= signal_series.shift(1))
    
    # FIX LOOKAHEAD BIAS: Shift signals by 1 bar
    entries = entries_raw.shift(1).fillna(False).astype(bool)
    exits = exits_raw.shift(1).fillna(False).astype(bool)
    
    # Run backtest
    pf = vbt.Portfolio.from_signals(
        close=close_series.to_numpy(dtype=float),
        entries=entries.to_numpy(dtype=bool),
        exits=exits.to_numpy(dtype=bool),
        init_cash=init_cash,
        fees=0.0005,
        slippage=0.0005,
        freq=freq
    )
    
    return pf

def collect_metrics(pf, fast, slow, signal, freq=FREQ, init_cash=100_000):
    """
    Collect all metrics from a portfolio backtest.
    """
    # Basic metrics
    total_return = float(pf.total_return())
    total_profit = float(pf.total_profit())
    
    # Annualized metrics
    try:
        ann_return = float(pf.annualized_return())
    except:
        ann_return = np.nan
    
    # Risk metrics
    sharpe = float(pf.sharpe_ratio(freq=freq))
    sortino = float(pf.sortino_ratio(freq=freq))
    max_dd = float(pf.max_drawdown())
    volatility = float(pf.annualized_volatility(freq=freq))
    
    # Calmar ratio
    try:
        calmar = float(pf.calmar_ratio())
    except:
        calmar = np.nan
    
    # Omega ratio
    try:
        omega = float(pf.omega_ratio())
    except:
        omega = np.nan
    
    # Trade statistics
    trades = pf.trades
    total_trades = len(trades)
    
    if total_trades > 0:
        trade_returns = trades.returns.values if hasattr(trades.returns, 'values') else np.array(trades.returns)
        winning_trades = trade_returns[trade_returns > 0]
        losing_trades = trade_returns[trade_returns < 0]
        
        win_rate = len(winning_trades) / total_trades * 100 if total_trades > 0 else 0
        
        # Profit factor
        gross_profit = winning_trades.sum() if len(winning_trades) > 0 else 0
        gross_loss = abs(losing_trades.sum()) if len(losing_trades) > 0 else 0
        profit_factor = gross_profit / gross_loss if gross_loss > 0 else np.inf
        
        # Expectancy
        expectancy = float(trade_returns.mean()) if len(trade_returns) > 0 else 0
        
        # Payoff ratio
        avg_win = winning_trades.mean() if len(winning_trades) > 0 else 0
        avg_loss = abs(losing_trades.mean()) if len(losing_trades) > 0 else 0
        payoff_ratio = avg_win / avg_loss if avg_loss > 0 else np.inf
        
        # Largest win/loss
        largest_win = winning_trades.max() if len(winning_trades) > 0 else 0
        largest_loss = losing_trades.min() if len(losing_trades) > 0 else 0
        
        # Win/loss amounts
        avg_win_amount = avg_win * init_cash
        avg_loss_amount = avg_loss * init_cash
        
        # Streaks
        try:
            streak_data = np.where(trade_returns > 0, 1, -1)
            winning_streak = 0
            losing_streak = 0
            current_streak = 0
            for s in streak_data:
                if s == 1:
                    if current_streak > 0:
                        current_streak += 1
                    else:
                        current_streak = 1
                    winning_streak = max(winning_streak, current_streak)
                else:
                    if current_streak < 0:
                        current_streak -= 1
                    else:
                        current_streak = -1
                    losing_streak = max(losing_streak, abs(current_streak))
        except:
            winning_streak = 0
            losing_streak = 0
        
        # SQN (System Quality Number)
        sqn = np.sqrt(total_trades) * expectancy / trade_returns.std() if trade_returns.std() > 0 else 0
        
        # Average trade duration
        try:
            avg_duration = float(trades.duration.mean().total_seconds() / 86400) if hasattr(trades.duration.mean(), 'total_seconds') else float(trades.duration.mean())
        except:
            avg_duration = np.nan
    else:
        win_rate = 0
        profit_factor = 0
        expectancy = 0
        payoff_ratio = 0
        largest_win = 0
        largest_loss = 0
        avg_win_amount = 0
        avg_loss_amount = 0
        winning_streak = 0
        losing_streak = 0
        sqn = 0
        avg_duration = 0
    
    # Information ratio (vs benchmark = buy & hold)
    try:
        info_ratio = float(pf.information_ratio())
    except:
        info_ratio = np.nan
    
    # Tail ratio
    try:
        returns = pf.returns()
        tail_ratio = abs(np.percentile(returns, 95)) / abs(np.percentile(returns, 5)) if np.percentile(returns, 5) != 0 else np.nan
    except:
        tail_ratio = np.nan
    
    # Deflated Sharpe Ratio (simplified approximation)
    try:
        dsr = sharpe * np.sqrt(1 - (stats.skew(pf.returns()) * sharpe / 4) - ((stats.kurtosis(pf.returns()) - 3) * sharpe**2 / 24))
    except:
        dsr = np.nan
    
    # Ulcer Index
    try:
        equity = pf.value()
        running_max = equity.cummax()
        drawdown_pct = (equity - running_max) / running_max * 100
        ulcer_index = np.sqrt((drawdown_pct ** 2).mean())
    except:
        ulcer_index = np.nan
    
    # Recovery factor
    try:
        recovery_factor = total_return / abs(max_dd) if max_dd != 0 else np.inf
    except:
        recovery_factor = np.nan
    
    # Gain to Pain ratio
    try:
        returns = pf.returns()
        gain_to_pain = returns.sum() / abs(returns[returns < 0].sum()) if returns[returns < 0].sum() != 0 else np.inf
    except:
        gain_to_pain = np.nan
    
    # Serenity Index (simplified)
    try:
        serenity = (total_return / abs(max_dd)) / (volatility / 0.15) if volatility > 0 and max_dd != 0 else np.nan
    except:
        serenity = np.nan
    
    return {
        'fast_period': fast,
        'slow_period': slow,
        'signal_period': signal,
        'total_return': total_return,
        'annualized_return': ann_return,
        'total_profit': total_profit,
        'sharpe_ratio': sharpe,
        'sortino_ratio': sortino,
        'calmar_ratio': calmar,
        'omega_ratio': omega,
        'information_ratio': info_ratio,
        'tail_ratio': tail_ratio,
        'deflated_sharpe_ratio': dsr,
        'max_drawdown': max_dd,
        'volatility': volatility,
        'ulcer_index': ulcer_index,
        'win_rate': win_rate,
        'total_trades': total_trades,
        'avg_trade_duration': avg_duration,
        'expectancy': expectancy,
        'profit_factor': profit_factor,
        'sqn': sqn,
        'payoff_ratio': payoff_ratio,
        'largest_win': largest_win,
        'largest_loss': largest_loss,
        'avg_win_amount': avg_win_amount,
        'avg_loss_amount': avg_loss_amount,
        'winning_streak': winning_streak,
        'losing_streak': losing_streak,
        'recovery_factor': recovery_factor,
        'gain_to_pain_ratio': gain_to_pain,
        'serenity_index': serenity
    }

# Run grid search
print(f"Starting MACD Grid Search over {len(macd_combinations)} combinations...")
print("="*60)

for fast, slow, signal in tqdm(macd_combinations, desc="Testing MACD combinations"):
    try:
        pf = run_macd_backtest(fast, slow, signal, train_close)
        metrics = collect_metrics(pf, fast, slow, signal)
        grid_search_results.append(metrics)
    except Exception as e:
        # Skip failed combinations
        pass

print(f"\nCompleted! Tested {len(grid_search_results)} combinations successfully.")

In [ ]:
# CONVERT RESULTS TO DATAFRAME AND ANALYZE

results_df = pd.DataFrame(grid_search_results)

# Sort by Sharpe ratio
results_df = results_df.sort_values('sharpe_ratio', ascending=False).reset_index(drop=True)

print(f"Total combinations tested: {len(results_df)}")
print("\n" + "="*80)
print("TOP 10 MACD STRATEGIES BY SHARPE RATIO (TRAINING SET)")
print("="*80)

# Display top 10
top_cols = ['fast_period', 'slow_period', 'signal_period', 'sharpe_ratio', 'sortino_ratio', 
            'total_return', 'max_drawdown', 'win_rate', 'total_trades', 'profit_factor']
print(results_df[top_cols].head(10).to_string())

# Best strategy summary
best = results_df.iloc[0]
print("\n" + "="*80)
print(f"BEST MACD STRATEGY: Fast={int(best['fast_period'])}, Slow={int(best['slow_period'])}, Signal={int(best['signal_period'])}")
print("="*80)
print(f"  Sharpe Ratio: {best['sharpe_ratio']:.4f}")
print(f"  Sortino Ratio: {best['sortino_ratio']:.4f}")
print(f"  Total Return: {best['total_return']*100:.2f}%")
print(f"  Max Drawdown: {best['max_drawdown']*100:.2f}%")
print(f"  Win Rate: {best['win_rate']:.2f}%")
print(f"  Total Trades: {int(best['total_trades'])}")
print(f"  Profit Factor: {best['profit_factor']:.4f}")

In [ ]:
# VALIDATION SET BACKTEST WITH BEST PARAMETERS

best = results_df.iloc[0]
fast_best = int(best['fast_period'])
slow_best = int(best['slow_period'])
signal_best = int(best['signal_period'])

print(f"Running validation backtest with MACD({fast_best},{slow_best},{signal_best})...")
print("="*60)

# Run on validation set
pf_val = run_macd_backtest(fast_best, slow_best, signal_best, val_close)
val_metrics = collect_metrics(pf_val, fast_best, slow_best, signal_best)

print("\nVALIDATION SET RESULTS:")
print(f"  Total Return: {val_metrics['total_return']*100:.2f}%")
print(f"  Sharpe Ratio: {val_metrics['sharpe_ratio']:.4f}")
print(f"  Sortino Ratio: {val_metrics['sortino_ratio']:.4f}")
print(f"  Max Drawdown: {val_metrics['max_drawdown']*100:.2f}%")
print(f"  Win Rate: {val_metrics['win_rate']:.2f}%")
print(f"  Total Trades: {int(val_metrics['total_trades'])}")
print(f"  Profit Factor: {val_metrics['profit_factor']:.4f}")

# Compare train vs validation
print("\n" + "="*60)
print("TRAIN VS VALIDATION COMPARISON:")
print("="*60)
print(f"{'Metric':<25} {'Train':>15} {'Validation':>15}")
print("-"*60)
print(f"{'Sharpe Ratio':<25} {best['sharpe_ratio']:>15.4f} {val_metrics['sharpe_ratio']:>15.4f}")
print(f"{'Total Return':<25} {best['total_return']*100:>14.2f}% {val_metrics['total_return']*100:>14.2f}%")
print(f"{'Max Drawdown':<25} {best['max_drawdown']*100:>14.2f}% {val_metrics['max_drawdown']*100:>14.2f}%")
print(f"{'Win Rate':<25} {best['win_rate']:>14.2f}% {val_metrics['win_rate']:>14.2f}%")
print(f"{'Total Trades':<25} {int(best['total_trades']):>15d} {int(val_metrics['total_trades']):>15d}")

In [ ]:
# FULL-SAMPLE BACKTEST AND VISUALIZATION

if 'FREQ' not in globals():
    FREQ = "1D"

# Best params from grid search
best = results_df.loc[results_df['sharpe_ratio'].idxmax()]
fast_best = int(best['fast_period'])
slow_best = int(best['slow_period'])
signal_best = int(best['signal_period'])

print(f"Running full-sample backtest with MACD({fast_best},{slow_best},{signal_best})...")

# Full sample close series
full_close = close.astype(float)

# Compute MACD
macd_line, signal_line = compute_macd(full_close, fast_best, slow_best, signal_best)

# Generate signals
entries_raw = (macd_line > signal_line) & (macd_line.shift(1) <= signal_line.shift(1))
exits_raw = (macd_line < signal_line) & (macd_line.shift(1) >= signal_line.shift(1))

# FIX LOOKAHEAD BIAS: Shift signals by 1 bar
entries_full = entries_raw.shift(1).fillna(False).astype(bool)
exits_full = exits_raw.shift(1).fillna(False).astype(bool)

# Run backtest
pf_full = vbt.Portfolio.from_signals(
    close=full_close.to_numpy(dtype=float),
    entries=entries_full.to_numpy(dtype=bool),
    exits=exits_full.to_numpy(dtype=bool),
    init_cash=100_000,
    fees=0.0005,
    slippage=0.0005,
    freq=FREQ
)

print(f"Full-sample backtest completed!")
print(f"  Total Return: {pf_full.total_return()*100:.2f}%")
print(f"  Sharpe Ratio: {pf_full.sharpe_ratio():.4f}")
print(f"  Max Drawdown: {pf_full.max_drawdown()*100:.2f}%")

In [ ]:
# EQUITY CURVE VISUALIZATION

fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)

# Plot 1: Price with MACD signals
ax1 = axes[0]
ax1.plot(full_close.index, full_close.values, label='Price', color='black', linewidth=1)

# Mark entry/exit points
entry_idx = entries_full[entries_full].index
exit_idx = exits_full[exits_full].index
ax1.scatter(entry_idx, full_close.loc[entry_idx], marker='^', color='green', s=100, label='Buy', zorder=5)
ax1.scatter(exit_idx, full_close.loc[exit_idx], marker='v', color='red', s=100, label='Sell', zorder=5)

ax1.set_title(f'Price with MACD({fast_best},{slow_best},{signal_best}) Signals', fontsize=14, fontweight='bold')
ax1.set_ylabel('Price')
ax1.legend(loc='upper left')
ax1.grid(True, alpha=0.3)

# Plot 2: MACD
ax2 = axes[1]
ax2.plot(full_close.index, macd_line.values, label='MACD Line', color='blue', linewidth=1)
ax2.plot(full_close.index, signal_line.values, label='Signal Line', color='orange', linewidth=1)
ax2.bar(full_close.index, (macd_line - signal_line).values, label='Histogram', color='gray', alpha=0.3, width=1)
ax2.axhline(0, color='black', linewidth=0.5, linestyle='--')
ax2.set_title('MACD Indicator', fontsize=14, fontweight='bold')
ax2.set_ylabel('MACD')
ax2.legend(loc='upper left')
ax2.grid(True, alpha=0.3)

# Plot 3: Equity Curve
ax3 = axes[2]
equity = pf_full.value()
ax3.plot(full_close.index, equity, label='Strategy Equity', color='blue', linewidth=2)

# Buy & Hold comparison
bh_equity = 100_000 * (full_close / full_close.iloc[0])
ax3.plot(full_close.index, bh_equity, label='Buy & Hold', color='gray', linewidth=1, linestyle='--')

ax3.set_title('Equity Curve: Strategy vs Buy & Hold', fontsize=14, fontweight='bold')
ax3.set_xlabel('Date')
ax3.set_ylabel('Portfolio Value ($)')
ax3.legend(loc='upper left')
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# FULL-SAMPLE TRADE-BY-TRADE RETURNS (BAR) + CUMULATIVE PER-TRADE EQUITY — ALL TRADES

if 'FREQ' not in globals():
    FREQ = "1D"

# Trade-by-trade returns (ALL trades)
trades = pf_full.trades
trade_returns = trades.returns.values if hasattr(trades.returns, 'values') else np.asarray(trades.returns)
trade_returns = np.asarray(trade_returns).ravel()  # ensure 1D

if trade_returns.size == 0:
    print("No trades to plot.")
else:
    # Calculate statistics
    winning_trades = trade_returns[trade_returns > 0]
    losing_trades = trade_returns[trade_returns < 0]
    
    total_trades = len(trade_returns)
    win_count = len(winning_trades)
    loss_count = len(losing_trades)
    win_rate = (win_count / total_trades * 100) if total_trades > 0 else 0
    
    avg_win_pct = (winning_trades.mean() * 100) if len(winning_trades) > 0 else 0
    avg_loss_pct = (losing_trades.mean() * 100) if len(losing_trades) > 0 else 0
    max_win_pct = (winning_trades.max() * 100) if len(winning_trades) > 0 else 0
    max_loss_pct = (losing_trades.min() * 100) if len(losing_trades) > 0 else 0
    
    print(f"Total trades plotted: {total_trades}")
    print(f"Win Rate: {win_rate:.1f}% ({win_count}W / {loss_count}L)")
    print(f"Avg Win: {avg_win_pct:.2f}% | Avg Loss: {avg_loss_pct:.2f}%")
    print(f"Max Win: {max_win_pct:.2f}% | Max Loss: {max_loss_pct:.2f}%")
    
    equity_per_trade = np.cumprod(1.0 + trade_returns)

    fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=False)

    # Per-trade returns (%), all trades
    x = np.arange(1, trade_returns.size + 1)
    colors = np.where(trade_returns >= 0, 'green', 'red')
    axes[0].bar(x, trade_returns * 100.0, color=colors, alpha=0.85, width=0.8)
    axes[0].axhline(0, color='black', linewidth=1, alpha=0.6)
    
    # Add statistics text box on the chart
    stats_text = (
        f'Win Rate: {win_rate:.1f}% ({win_count}W/{loss_count}L)\n'
        f'Avg Win: {avg_win_pct:.2f}% | Avg Loss: {avg_loss_pct:.2f}%\n'
        f'Max Win: {max_win_pct:.2f}% | Max Loss: {max_loss_pct:.2f}%'
    )
    axes[0].text(0.02, 0.98, stats_text, transform=axes[0].transAxes,
                fontsize=10, verticalalignment='top',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
    
    axes[0].set_title(f'Per-Trade Returns (%) - MACD({fast_best},{slow_best},{signal_best}) Full Sample', 
                     fontsize=13, fontweight='bold')
    axes[0].set_ylabel('Return (%)')
    axes[0].grid(True, alpha=0.3)

    # Cumulative equity per trade
    axes[1].plot(x, equity_per_trade, color='black', linewidth=2)
    axes[1].set_title('Cumulative Equity per Trade (Trade Domain)', fontsize=13)
    axes[1].set_xlabel('Trade #')
    axes[1].set_ylabel('Equity (x)')
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


In [ ]:
#PARAMETER SENSITIVITY TABLE

if results_df.empty:
    print("No results available for sensitivity analysis.")
else:
    # Get BEST strategy (highest Sharpe ratio)
    best = results_df.loc[results_df['sharpe_ratio'].idxmax()]
    fast_best = int(best['fast_period'])
    slow_best = int(best['slow_period'])
    signal_best = int(best['signal_period'])

    print(f"Parameter Sensitivity Analysis for Best MACD({fast_best},{slow_best},{signal_best})")
    print("=" * 80)

    # Create sensitivity ranges (±5 around each parameter)
    fast_candidates = list(range(max(2, fast_best - 5), fast_best + 6))
    slow_candidates = list(range(max(3, slow_best - 5), slow_best + 6))
    signal_candidates = list(range(max(2, signal_best - 5), signal_best + 6))
    
    # Test variations: vary one parameter at a time
    combos = ([(f, slow_best, signal_best) for f in fast_candidates if f < slow_best] +
              [(fast_best, s, signal_best) for s in slow_candidates if s != slow_best and fast_best < s] +
              [(fast_best, slow_best, sig) for sig in signal_candidates if sig != signal_best])

    def eval_combo(fast: int, slow: int, signal: int) -> dict:
        try:
            pf = run_macd_backtest(fast, slow, signal, train_close)
            
            total_return = float(pf.total_return())
            sharpe = float(pf.sharpe_ratio(freq=FREQ))
            sortino = float(pf.sortino_ratio(freq=FREQ))
            mdd = float(pf.max_drawdown())
            vol = float(pf.annualized_volatility(freq=FREQ))

            trades = pf.trades
            ntr = len(trades)
            win_rate_pct = np.nan
            profit_factor = np.nan
            expectancy = 0.0
            if ntr > 0:
                tr = trades.returns.values if hasattr(trades.returns, 'values') else np.array(trades.returns)
                if tr.size > 0:
                    pos = tr[tr > 0]
                    neg = tr[tr < 0]
                    win_rate_pct = (len(pos) / len(tr)) * 100 if len(tr) else np.nan
                    gains = pos.sum() if len(pos) else 0.0
                    losses = abs(neg.sum()) if len(neg) else 0.0
                    profit_factor = gains / losses if losses > 0 else np.inf
                    expectancy = float(tr.mean())

            return {
                'fast': fast, 'slow': slow, 'signal': signal,
                'sharpe': sharpe, 'sortino': sortino,
                'total_return': total_return, 'max_drawdown': mdd, 'volatility': vol,
                'total_trades': ntr, 'win_rate_pct': win_rate_pct,
                'profit_factor': profit_factor, 'expectancy': expectancy
            }
        except:
            return None

    rows = []
    for combo in combos:
        result = eval_combo(*combo)
        if result is not None:
            rows.append(result)

    if not rows:
        print("No sensitivity results computed.")
    else:
        sens = pd.DataFrame(rows)

        # Show as a table
        cols = ['fast','slow','signal','sharpe','sortino','total_return','max_drawdown','volatility',
                'total_trades','win_rate_pct','profit_factor','expectancy']
        sens_table = sens[cols].sort_values(['fast','slow','signal'])
        
        print(f"Sensitivity Results ({len(sens_table)} variations tested):\n")
        display(sens_table)

        # Compact variation summary
        metric_cols = ['sharpe','sortino','total_return','max_drawdown','volatility',
                       'win_rate_pct','profit_factor','expectancy']
        summary = sens_table[metric_cols].agg(['mean','std','min','max']).T
        
        print("Sensitivity Summary (mean / std / min / max):")
        print(summary.round(4).to_string())

